# Liver Disease — Model Training
Trains and compares models, saves the best as
`models/liver_disease_model.pkl`.

In [1]:
import pandas as pd
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed — skipping it.")


In [2]:
X_train = pd.read_csv("../../data/processed/liver_disease_X_train.csv")
X_test = pd.read_csv("../../data/processed/liver_disease_X_test.csv")
y_train = pd.read_csv("../../data/processed/liver_disease_y_train.csv").squeeze()
y_test = pd.read_csv("../../data/processed/liver_disease_y_test.csv").squeeze()


In [3]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}
if HAS_XGB:
    models["XGBoost"] = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)


In [4]:
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    })

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
results_df


c:\Users\SADAB EHTESHAM\Desktop\m project\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [22:39:19] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.735043,0.740741,0.963855,0.837696,0.830617
2,Random Forest,0.752137,0.775510,0.915663,0.839779,0.747697
3,XGBoost,0.700855,0.744898,0.879518,0.806630,0.727853
1,Decision Tree,0.598291,0.709302,0.734940,0.721893,0.499823


In [5]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
print(f"Best model: {best_model_name}")

y_pred_best = best_model.predict(X_test)
print(confusion_matrix(y_test, y_pred_best))


Best model: Logistic Regression
[[ 6 28]
 [ 3 80]]


In [6]:
joblib.dump(best_model, "../../models/liver_disease_model.pkl")
print("Saved: models/liver_disease_model.pkl")


Saved: models/liver_disease_model.pkl


## Next step
Open **04_evaluation.ipynb**.